# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa - Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint
from IPython.display import display

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a Dataset object, not a dict
print(f"Dataset Name: {getattr(metadata, 'name', '<no name found>')}")
print(f"Description: {getattr(metadata, 'description', '<no description found>')}")


## 2. Data Overview

Review available record sets and their `@id` fields, plus a summary of their fields and columns.


In [ ]:
# List all record sets and their information

record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in this dataset.")
else:
    print("Available record sets in the dataset:")
    for rset in record_sets:
        # Each record set object in mlcroissant exposes its @id and name
        print(f"  - @id: {getattr(rset, '@id', None)} | name: {getattr(rset, 'name', '')}")
        # List fields/columns in each record set
        print("    Fields (with @id):")
        for fld in getattr(rset, 'fields', []):
            print(f"     - {getattr(fld, '@id', None)} (name: {getattr(fld, 'name', '')}, dataType: {getattr(fld, 'data_type', '')})")


## 3. Data Extraction

Load data from the main record set into a DataFrame for analysis. All entity references are by their `@id` fields as per Croissant specification.


In [ ]:
# For this dataset, there is typically a primary record set for survey responses.
# List all record set @ids:
record_sets = list(dataset.record_sets)
# We'll use the first (and usually only) record set if only one exists:
if record_sets:
    main_record_set = record_sets[0]
    main_record_set_id = getattr(main_record_set, '@id')
    print(f"Using primary record set @id: {main_record_set_id}")
else:
    raise ValueError("No record sets found!")

# Extract all records and load into a DataFrame
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print("Columns in extracted DataFrame:")
print(df.columns.tolist())
display(df.head())

## 4. Exploratory Data Analysis (EDA)

We will examine a numeric field (e.g., GAD-7 score, typically used in mental health survey data)---referenced by its field `@id`.

Let's display summary statistics, filter by a threshold, normalize, and group by a demographic variable (e.g., gender, referenced by its `@id`).


In [ ]:
# Use the metadata inspection above to find field @ids
# For this example, we presume GAD-7 scores and gender are columns available
# Replace with actual @ids found above if different
gad7_field_id = 'gad7_score'        # <--- update to actual @id if different (e.g., 'cr:field/gad7_score')
gender_field_id = 'gender'          # <--- update to actual @id if different

if gad7_field_id in df.columns:
    print(f"Summary statistics for {gad7_field_id}:\n", df[gad7_field_id].describe())
    # Filter records where GAD-7 > 10
    threshold = 10
    filtered_df = df[df[gad7_field_id] > threshold].copy()
    print(f"\nRecords with {gad7_field_id} > {threshold}: {len(filtered_df)}")

    if not filtered_df.empty:
        # Normalize
        filtered_df[f"{gad7_field_id}_normalized"] = (filtered_df[gad7_field_id] - filtered_df[gad7_field_id].mean()) / filtered_df[gad7_field_id].std()
        display(filtered_df[[gad7_field_id, f"{gad7_field_id}_normalized"]].head())

        # Group by gender
        if gender_field_id in filtered_df.columns:
            grouped = filtered_df.groupby(gender_field_id)[gad7_field_id].mean()
            print(f"\nAverage {gad7_field_id} by {gender_field_id} for filtered records:")
            display(grouped)
        else:
            print(f"Field '{gender_field_id}' not present in DataFrame. Columns: {df.columns.tolist()}")
    else:
        print(f"No records meet the filter ({gad7_field_id} > {threshold}).")
else:
    print(f"Field '{gad7_field_id}' not found in columns: {df.columns.tolist()}")

## 5. Visualization

Let's visualize the distribution of GAD-7 scores and compare them by gender.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of GAD-7 scores
if gad7_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[gad7_field_id].dropna(), kde=True, bins=8, color='skyblue')
    plt.title('Distribution of GAD-7 Scores')
    plt.xlabel('GAD-7 Score')
    plt.ylabel('Number of Participants')
    plt.show()

    # Boxplot by gender, if available
    if gender_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=gender_field_id, y=gad7_field_id, data=df)
        plt.title('GAD-7 Scores by Gender')
        plt.xlabel('Gender')
        plt.ylabel('GAD-7 Score')
        plt.show()


## 6. Conclusion

- We demonstrated how to load and explore a Croissant dataset using the `mlcroissant` library.
- The FAIR² Mental Health Survey Dataset provides GAD-7, PHQ-9, PSQ, and demographic information. We showed how to extract, filter, and analyze numeric fields (GAD-7) and their distribution by demographic variables.
- Such AI-ready, well-documented datasets support reproducible research and downstream analysis for health and social science.
